In [ ]:
!pip -q install datasets

from datasets import load_dataset

dataset = load_dataset("lukebarousse/data_jobs", split="train")
df = dataset.to_pandas()
df = df.drop_duplicates().copy()  # Remove exact duplicate postings

print("Rows and columns:", df.shape)
print(df.head())
print(df.columns.tolist())

# Data Job Market Analysis: Skills, Roles, and Remote Work

## Goal
Explore 2023 job postings to see which skills appear most often in data analyst and data scientist roles, how the requirements compare, and how many postings are marked as remote.

## Questions
1. Which skills appear most often in each role?
2. How do skill requirements differ between roles?
3. What share of postings for each role are marked as remote?
4. Which countries are most represented in this dataset?

## Data
Luke Barousse’s data jobs dataset, containing job postings from 2023. This is a historical dataset, so the results describe this sample—not the current job market.

## Method
Remove exact duplicate rows, select analyst and scientist roles, and keep postings with non-empty skill information. Count each skill once per posting, then compare the share of postings mentioning it. The remote-work and country summaries use this same skill-bearing subset.

In [ ]:
print("Dataset size:", df.shape)

print("\nColumn types:")
print(df.dtypes)

print("\nMissing values in the most useful columns:")
columns_to_check = [
    "job_title_short",
    "job_country",
    "job_work_from_home",
    "salary_year_avg",
    "job_skills",
]
print(df[columns_to_check].isna().sum())

print("\nDuplicate rows:", df.duplicated().sum())

print("\nMost common job categories:")
print(df["job_title_short"].value_counts().head(10))

print("\nExample skills values:")
print(df["job_skills"].dropna().head(5).tolist())

## Skill demand by role

For each role, I count how many postings mention each skill and calculate that as a percentage of the postings in that role. This makes the comparison fairer when the roles have different numbers of postings.

In [ ]:
import ast
import pandas as pd
import matplotlib.pyplot as plt

target_roles = [
    "Data Analyst",
    "Senior Data Analyst",
    "Data Scientist",
    "Senior Data Scientist",
]

jobs = df[df["job_title_short"].isin(target_roles)].copy()

def to_skill_list(value):
    if isinstance(value, list):
        return value
    if pd.isna(value):
        return []
    try:
        parsed = ast.literal_eval(value)
        return parsed if isinstance(parsed, list) else []
    except (ValueError, SyntaxError):
        return []

jobs["skills_list"] = jobs["job_skills"].apply(to_skill_list)
jobs = jobs[jobs["skills_list"].str.len() > 0].copy()

# Give each posting an ID so repeated skills within a posting count once
jobs = jobs.reset_index(drop=True)
jobs["job_id"] = jobs.index

skills = jobs[["job_id", "job_title_short", "skills_list"]].explode("skills_list")
skills = skills.rename(columns={"skills_list": "skill"})
skills = skills.drop_duplicates(subset=["job_id", "skill"])

skill_counts = (
    skills.groupby(["job_title_short", "skill"])
    .size()
    .rename("postings")
    .reset_index()
)

role_totals = jobs.groupby("job_title_short").size()
skill_counts["percent_of_postings"] = skill_counts.apply(
    lambda row: 100 * row["postings"] / role_totals[row["job_title_short"]],
    axis=1,
)

top_skills = (
    skill_counts.groupby("skill")["percent_of_postings"]
    .mean()
    .nlargest(12)
    .index
)

chart_data = (
    skill_counts[skill_counts["skill"].isin(top_skills)]
    .pivot(index="skill", columns="job_title_short", values="percent_of_postings")
    .fillna(0)
)

chart_data.plot(kind="barh", figsize=(10, 7))
plt.title("Most requested skills by data job role")
plt.xlabel("Share of job postings mentioning the skill (%)")
plt.ylabel("")
plt.tight_layout()
plt.show()

print("Usable postings:", len(jobs))
print(chart_data.round(1))

## Remote work by role

Compare the share of postings marked as remote for each role. These figures describe only the postings in this dataset.

In [ ]:
remote_summary = (
    jobs.groupby("job_title_short")
    .agg(
        postings=("job_work_from_home", "size"),
        remote_percent=("job_work_from_home", lambda x: 100 * x.mean()),
    )
    .sort_values("remote_percent", ascending=False)
)

print(remote_summary.round(1))

remote_summary["remote_percent"].plot(
    kind="bar",
    figsize=(8, 5),
    color="teal",
)
plt.title("Share of postings marked as remote")
plt.ylabel("Remote postings (%)")
plt.xlabel("")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

## Countries represented

Show the countries with the most postings for each role. These are counts in this dataset, not a ranking of the overall job market.

In [ ]:
country_counts = (
    jobs.dropna(subset=["job_country"])
    .groupby(["job_title_short", "job_country"])
    .size()
    .rename("postings")
    .reset_index()
)

for role in country_counts["job_title_short"].unique():
    print(f"\nTop countries for {role}:")
    print(
        country_counts[country_counts["job_title_short"] == role]
        .nlargest(5, "postings")[["job_country", "postings"]]
        .to_string(index=False)
    )

## Summary tables

In [ ]:
for role in skill_counts["job_title_short"].unique():
    print(f"\nTop skills for {role}:")
    print(
        skill_counts[skill_counts["job_title_short"] == role]
        .nlargest(5, "percent_of_postings")[
            ["skill", "postings", "percent_of_postings"]
        ]
        .round(1)
        .to_string(index=False)
    )

print("\nRemote share by role (%):")
print(remote_summary[["postings", "remote_percent"]].round(1))